In [41]:
import warnings
from pprint import pprint

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

# plotting style
plt.style.use("seaborn-v0_8")
sns.set_palette("tab10")
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

In [42]:
device = "kria_openssl"
device = "kria_swtpm"
device = "kria_tpm"
device = "rpi_openssl"
device = "rpi_swtpm"
# device = "ubuntu_openssl"
# device = "ubuntu_swtpm"

In [43]:
results_path = "./data/{device}/results_{i}.csv"
telemetry_df = pd.read_csv(f"./data/{device}/resource_usage.csv")
telemetry_df["timestamp"] = pd.to_datetime(telemetry_df["timestamp"])

results_df_list = []
run_times = {}

# read and preprocess results
for i in range(1, 6):
    df = pd.read_csv(results_path.format(device=device, i=i))
    df["run"] = i

    df["integrity_start"] = pd.to_datetime(df["integrity_start"])
    df["integrity_end"] = pd.to_datetime(df["integrity_end"])
    df["inference_start"] = pd.to_datetime(df["inference_start"])
    df["inference_end"] = pd.to_datetime(df["inference_end"])

    df["processing_time_ms"] = df["integrity_duration_ms"] + df["inference_duration_ms"]

    # wi = with integrity check
    # woi = without integrity check
    run_times[i] = {}
    run_times[i]["wi_start"] = df[df["integrity_check_enabled"] == True]["inference_end"].min().to_datetime64()
    run_times[i]["wi_end"] = df[df["integrity_check_enabled"] == True]["inference_end"].max().to_datetime64()

    run_times[i]["woi_start"] = df[df["integrity_check_enabled"] == False]["inference_end"].min().to_datetime64()
    run_times[i]["woi_end"] = df[df["integrity_check_enabled"] == False]["inference_end"].max().to_datetime64()

    df["inference_end"].min().to_datetime64()

    results_df_list.append(df)
    del df

results_df: pd.DataFrame = pd.concat(results_df_list).sort_values(
    by=["run", "image_name", "integrity_check_enabled"], ignore_index=True
)

print("Inference DataFrame shape:", results_df.shape)
# print("Inference columns:", results_df.info())
print("Inference start   :", results_df["integrity_start"].min().to_datetime64())
print("Inference end     :", results_df["inference_end"].max().to_datetime64())
print("\nTelemetry DataFrame shape:", telemetry_df.shape)
# print("Telemetry columns :", telemetry_df.info())
print("Telemetry start   :", telemetry_df["timestamp"].min().to_datetime64())
print("Telemetry end     :", telemetry_df["timestamp"].max().to_datetime64())
print("\nRun times:")
pprint(run_times, sort_dicts=False)

Inference DataFrame shape: (420, 12)
Inference start   : 2025-07-24T12:12:18.488097000
Inference end     : 2025-07-24T12:26:44.223571000

Telemetry DataFrame shape: (94131, 4)
Telemetry start   : 2025-07-24T12:12:00.983000000
Telemetry end     : 2025-07-24T12:27:42.276000000

Run times:
{1: {'wi_start': numpy.datetime64('2025-07-24T12:12:19.493778000'),
     'wi_end': numpy.datetime64('2025-07-24T12:12:53.813637000'),
     'woi_start': numpy.datetime64('2025-07-24T12:13:54.620182000'),
     'woi_end': numpy.datetime64('2025-07-24T12:14:23.805272000')},
 2: {'wi_start': numpy.datetime64('2025-07-24T12:15:24.740168000'),
     'wi_end': numpy.datetime64('2025-07-24T12:15:58.840999000'),
     'woi_start': numpy.datetime64('2025-07-24T12:16:59.615564000'),
     'woi_end': numpy.datetime64('2025-07-24T12:17:28.973530000')},
 3: {'wi_start': numpy.datetime64('2025-07-24T12:18:29.869682000'),
     'wi_end': numpy.datetime64('2025-07-24T12:19:04.011294000'),
     'woi_start': numpy.datetime64('

In [44]:
wi_telemetry_list = []
woi_telemetry_list = []

for run_id, times in run_times.items():
    print(f"Run {run_id}:")
    print(f"  With Integrity Check    :: Start: {times['wi_start']}, End: {times['wi_end']}")
    print(f"  Without Integrity Check :: Start: {times['woi_start']}, End: {times['woi_end']}")

    wi_data = telemetry_df[
        (telemetry_df["timestamp"] >= times["wi_start"]) & (telemetry_df["timestamp"] <= times["wi_end"])
    ]
    wi_data["run"] = run_id
    wi_data["integrity_check_enabled"] = True
    wi_telemetry_list.append(wi_data)

    woi_data = telemetry_df[
        (telemetry_df["timestamp"] >= times["woi_start"]) & (telemetry_df["timestamp"] <= times["woi_end"])
    ]
    woi_data["run"] = run_id
    woi_data["integrity_check_enabled"] = False
    woi_telemetry_list.append(woi_data)

# concatenate telemetry for all runs
wi_telemetry_all = pd.concat(wi_telemetry_list, ignore_index=True)
woi_telemetry_all = pd.concat(woi_telemetry_list, ignore_index=True)

print(f"\nTotal WI telemetry rows: {len(wi_telemetry_all)}")
print(f"Total WOI telemetry rows: {len(woi_telemetry_all)}")

Run 1:
  With Integrity Check    :: Start: 2025-07-24T12:12:19.493778000, End: 2025-07-24T12:12:53.813637000
  Without Integrity Check :: Start: 2025-07-24T12:13:54.620182000, End: 2025-07-24T12:14:23.805272000
Run 2:
  With Integrity Check    :: Start: 2025-07-24T12:15:24.740168000, End: 2025-07-24T12:15:58.840999000
  Without Integrity Check :: Start: 2025-07-24T12:16:59.615564000, End: 2025-07-24T12:17:28.973530000
Run 3:
  With Integrity Check    :: Start: 2025-07-24T12:18:29.869682000, End: 2025-07-24T12:19:04.011294000
  Without Integrity Check :: Start: 2025-07-24T12:20:04.806334000, End: 2025-07-24T12:20:34.090564000
Run 4:
  With Integrity Check    :: Start: 2025-07-24T12:21:35.005968000, End: 2025-07-24T12:22:09.129316000
  Without Integrity Check :: Start: 2025-07-24T12:23:09.913797000, End: 2025-07-24T12:23:39.114270000
Run 5:
  With Integrity Check    :: Start: 2025-07-24T12:24:40.006100000, End: 2025-07-24T12:25:14.129863000
  Without Integrity Check :: Start: 2025-07-24T

In [45]:
# save integrity, inference and total processing time in a CSV file
from os import makedirs

makedirs("./data/processed_telemetry", exist_ok=True)

pd.concat([wi_telemetry_all, woi_telemetry_all])[
    [
        "run",
        "timestamp",
        "cpu_usage_percent",
        "memory_used_mb",
        "memory_usage_percent",
        "integrity_check_enabled",
    ]
].to_csv(f"./data/processed_telemetry/{device}.csv", index=False)

In [46]:
with_integrity = results_df[results_df["integrity_check_enabled"] == True]
without_integrity = results_df[results_df["integrity_check_enabled"] == False]

# total processing time
with_integrity_total = with_integrity["processing_time_ms"]
without_integrity_total = without_integrity["inference_duration_ms"]

print(f"=== Device: {device} ===")

# summary
print("\n=== Processing Time ===")
with_integrity_mean = with_integrity_total.mean()
without_integrity_mean = without_integrity_total.mean()
time_overhead = with_integrity_mean - without_integrity_mean
time_overhead_pct = (time_overhead / without_integrity_mean) * 100

print(f"Average total processing time with integrity: {with_integrity_mean:.2f} ms")
print(f"Average total processing time without integrity: {without_integrity_mean:.2f} ms")
print(f"Time overhead from integrity check: {time_overhead:.2f} ms ({time_overhead_pct:.1f}%)")
print(f"Average integrity check duration: {with_integrity['integrity_duration_ms'].mean():.2f} ms")


print("\n=== System Resource ===")
print(f"Average CPU usage with integrity: {wi_telemetry_all['cpu_usage_percent'].mean():.2f}%")
print(f"Average CPU usage without integrity: {woi_telemetry_all['cpu_usage_percent'].mean():.2f}%")
print(f"Average memory usage with integrity: {wi_telemetry_all['memory_usage_percent'].mean():.2f}%")
print(f"Average memory usage without integrity: {woi_telemetry_all['memory_usage_percent'].mean():.2f}%")

# peak resource usage
print("\n=== Peak Resource Usage ===")
print(f"Peak CPU with integrity: {wi_telemetry_all['cpu_usage_percent'].max():.2f}%")
print(f"Peak CPU without integrity: {woi_telemetry_all['cpu_usage_percent'].max():.2f}%")
print(f"Peak memory with integrity: {wi_telemetry_all['memory_usage_percent'].max():.2f}%")
print(f"Peak memory without integrity: {woi_telemetry_all['memory_usage_percent'].max():.2f}%")

=== Device: rpi_swtpm ===

=== Processing Time ===
Average total processing time with integrity: 834.34 ms
Average total processing time without integrity: 714.62 ms
Time overhead from integrity check: 119.72 ms (16.8%)
Average integrity check duration: 119.01 ms

=== System Resource ===
Average CPU usage with integrity: 37.54%
Average CPU usage without integrity: 39.60%
Average memory usage with integrity: 13.55%
Average memory usage without integrity: 13.60%

=== Peak Resource Usage ===
Peak CPU with integrity: 77.11%
Peak CPU without integrity: 78.78%
Peak memory with integrity: 17.44%
Peak memory without integrity: 17.06%
